In [ ]:
import pandas as pd
import glob
import time
import os

# 1. Instalacao da biblioteca
os.system('pip install git+https://github.com/GONZOsint/factcheckexplorer.git')
from factcheckexplorer.factcheckexplorer import FactCheckLib

# 2. Lista de palavras chave
palavras_chave = [
    "eleição", "eleições", "bolsonaro", "lula", "pt", "pl",
    "campanha", "urna", "voto", "fraude", "tse", "stf",
    "moraes", "exército", "intervenção", "apuração", "mesário"
]

print(" Iniciando a mineração de dados... (Isso pode levar alguns minutos)")

# 3. EXTRAÇÃO: Roda a biblioteca para gerar os CSVs temporários
for palavra in palavras_chave:
    try:
        fact_check = FactCheckLib(query=palavra, language="pt", num_results=100)
        fact_check.process()
        time.sleep(2) # Pausa para não sobrecarregar a API
        print(f" Notícias sobre '{palavra}' extraídas.")
    except Exception as e:
        print(f" Erro na palavra {palavra}: {e}")

# 4. CONSOLIDAÇÃO: Junta todos os arquivos em um Mega Dataset
arquivos_csv = glob.glob("*.csv")
lista_tabelas = [pd.read_csv(arquivo) for arquivo in arquivos_csv if 'MEGA' not in arquivo]

if lista_tabelas:
    df_mega = pd.concat(lista_tabelas, ignore_index=True)
    df_mega = df_mega.drop_duplicates(subset=['Claim']) # Remove notícias repetidas
    df_mega.to_csv("MEGA_dataset_eleicoes.csv", index=False)
    print(f"\n MINERAÇÃO CONCLUÍDA! Dataset consolidado salvo com {len(df_mega)} notícias verificadas.")
else:
    print(" Nenhum dado foi extraído.")

 Iniciando a mineração de dados... (Isso pode levar alguns minutos)
 Notícias sobre 'eleição' extraídas.
 Notícias sobre 'eleições' extraídas.
 Notícias sobre 'bolsonaro' extraídas.
 Notícias sobre 'lula' extraídas.
 Notícias sobre 'pt' extraídas.
 Notícias sobre 'pl' extraídas.
 Notícias sobre 'campanha' extraídas.
 Notícias sobre 'urna' extraídas.
 Notícias sobre 'voto' extraídas.
 Notícias sobre 'fraude' extraídas.
 Notícias sobre 'tse' extraídas.
 Erro na palavra stf: 'NoneType' object is not iterable
 Notícias sobre 'moraes' extraídas.
 Notícias sobre 'exército' extraídas.
 Notícias sobre 'intervenção' extraídas.
 Notícias sobre 'apuração' extraídas.
 Notícias sobre 'mesário' extraídas.

 MINERAÇÃO CONCLUÍDA! Dataset consolidado salvo com 1168 notícias verificadas.


In [ ]:
pip install feedparser


In [ ]:
#EXTRAIR NOTÍCIAS DO G1 PARA BALANCEAR O TREINAMENTO DO MODELO
import feedparser
import pandas as pd

print(" Extraindo notícias reais do G1...")

# URL do RSS de Política do G1
url_g1 = "https://g1.globo.com/rss/g1/politica/"
feed = feedparser.parse(url_g1)

# Pega as manchetes e os resumos
noticias_reais = []
for post in feed.entries:
    noticias_reais.append(post.title)
    if 'summary' in post:
        texto_limpo = post.summary.split('<')[0]
        noticias_reais.append(texto_limpo)

print(f" Conseguimos {len(noticias_reais)} frases reais do G1!")

# Transforma em um DataFrame (Tabela) marcando tudo como VERDADEIRO (1)
df_g1 = pd.DataFrame({
    'Claim': noticias_reais,
    'Verdict': ['verdade'] * len(noticias_reais),
    'Label': [1] * len(noticias_reais)
})

# Abre o nosso Mega Dataset que só tinha fake news
df_fake = pd.read_csv('MEGA_dataset_eleicoes.csv')
df_balanceado = pd.concat([df_fake, df_g1], ignore_index=True)
df_balanceado.to_csv('MEGA_dataset_eleicoes.csv', index=False)
print(" Dataset atualizado com notícias reais e salvo!")

 Extraindo notícias reais do G1...
✅ Conseguimos 200 frases reais do G1!
 Dataset atualizado com notícias reais e salvo!


In [ ]:
#MODELO SVM + AJUSTES (MAIS EFETIVO)
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

print(" Iniciando o treinamento do modelo AVANÇADO (SVM)...")

# 1. Carregar os dados
df = pd.read_csv('MEGA_dataset_eleicoes.csv')
df = df[['Claim', 'Verdict']].copy()
df['Verdict'] = df['Verdict'].astype(str).str.lower().str.strip()

# 2. Padronização Binária (0,1)
def padronizar_rotulo(verdict):
    if 'falso' in verdict or 'fake' in verdict or 'enganoso' in verdict or 'mentira' in verdict:
        return 0
    elif 'verdade' in verdict or 'fato' in verdict or 'true' in verdict:
        return 1
    else:
        return None

df['Label'] = df['Verdict'].apply(padronizar_rotulo)
df = df.dropna(subset=['Label'])

# 3. Injetando Fatos Verdadeiros (AGORA COM OS POLÍTICOS)
fatos_reais = [
    'Luiz Inácio Lula da Silva é o atual presidente do Brasil.',
    'Jair Bolsonaro foi presidente da república antes de 2023.',
    'As eleições ocorrerão no primeiro domingo de outubro.',
    'O voto é obrigatório para maiores de 18 anos no Brasil.',
    'Candidatos precisam registrar suas campanhas no Tribunal Superior Eleitoral.',
    'As urnas eletrônicas são utilizadas no sistema de votação brasileiro.',
    'O presidente da república cumpre um mandato de quatro anos.',
    'O TSE é o órgão responsável por organizar e fiscalizar as eleições.',
    'Candidatos participam de debates na televisão e rádio durante a campanha.',
    'O resultado das eleições começa a ser divulgado no mesmo dia da votação.',
    'Mesários são cidadãos convocados para trabalhar nas seções eleitorais.',
    'A propaganda eleitoral gratuita é garantida por lei.'
    'Deputados federais e senadores compõem o Congresso Nacional.',
    'A Constituição Federal é a lei máxima e rege o país.',
    'O mandato de um senador tem a duração de oito anos.',
    'Governadores e prefeitos administram os estados e municípios.'
]

multiplicador = 30
total_linhas = len(fatos_reais) * multiplicador

verdades_extras = pd.DataFrame({
    'Claim': fatos_reais * multiplicador,
    'Verdict': ['verdade'] * total_linhas,
    'Label': [1] * total_linhas
})
df = pd.concat([df, verdades_extras], ignore_index=True)

# 4. Limpeza de Texto (NLP)
def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    stopwords = ['o', 'a', 'os', 'as', 'um', 'uma', 'de', 'do', 'da', 'em', 'no', 'na', 'para', 'com', 'que', 'é', 'e', 'se', 'por']
    palavras = texto.split()
    palavras_limpas = [p for p in palavras if p not in stopwords]
    return ' '.join(palavras_limpas)

df['Claim_Limpo'] = df['Claim'].apply(limpar_texto)

# 5. Treinamento
X = df['Claim_Limpo']
y = df['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ⬅ MUDANÇA 3: Ativando os N-grams (lendo pares de palavras juntos)
vetorizador = TfidfVectorizer(ngram_range=(1, 2))
X_train_vetorizado = vetorizador.fit_transform(X_train)
X_test_vetorizado = vetorizador.transform(X_test)

# ⬅ MUDANÇA 4: Usando o modelo SVM
modelo = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
modelo.fit(X_train_vetorizado, y_train)

# 6. Resultados
previsoes = modelo.predict(X_test_vetorizado)
acuracia = accuracy_score(y_test, previsoes)

print("\n=============================================")
print(" RESULTADOS FINAIS DO MACHINE LEARNING (SVM)")
print("=============================================\n")
print(f" Acurácia: {acuracia * 100:.2f}%\n")
print(classification_report(y_test, previsoes, target_names=['Falso (0)', 'Verdadeiro (1)']))

# 7. Salvar os arquivos do Microserviço
joblib.dump(modelo, 'modelo_eleicoes.pkl')
joblib.dump(vetorizador, 'vetorizador.pkl')
print("\n Arquivos 'modelo_eleicoes.pkl' e 'vetorizador.pkl' salvos com sucesso!")

 Iniciando o treinamento do modelo AVANÇADO (SVM)...

 RESULTADOS FINAIS DO MACHINE LEARNING (SVM)

 Acurácia: 94.96%

                precision    recall  f1-score   support

     Falso (0)       0.94      0.99      0.96       208
Verdadeiro (1)       0.97      0.89      0.93       129

      accuracy                           0.95       337
     macro avg       0.96      0.94      0.95       337
  weighted avg       0.95      0.95      0.95       337


 Arquivos 'modelo_eleicoes.pkl' e 'vetorizador.pkl' salvos com sucesso!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import joblib

print(" Terminal de Teste da IA ativado!")
print("Digite 'sair' para encerrar o teste.\n")

modelo_teste = joblib.load('modelo_eleicoes.pkl')
vetorizador_teste = joblib.load('vetorizador.pkl')

while True:
    noticia = input(" Cole ou digite uma notícia aqui: ")

    if noticia.lower() == 'sair':
        print("Teste encerrado.")
        break

    texto_vetorizado = vetorizador_teste.transform([noticia])
    resultado = modelo_teste.predict(texto_vetorizado)[0]
    confianca = max(modelo_teste.predict_proba(texto_vetorizado)[0]) * 100

    if resultado == 1:
        print(f" Veredito: VERDADEIRO (Confiança: {confianca:.1f}%)\n")
    else:
        print(f" Veredito: FALSO/ENGANOSO (Confiança: {confianca:.1f}%)\n")

 Terminal de Teste da IA ativado!
Digite 'sair' para encerrar o teste.

 Cole ou digite uma notícia aqui: Lula é o atual presidente
 Veredito: VERDADEIRO (Confiança: 99.7%)

 Cole ou digite uma notícia aqui: so pode votar depois dos 18 anos
 Veredito: FALSO/ENGANOSO (Confiança: 73.8%)

 Cole ou digite uma notícia aqui: Lula rouba
 Veredito: FALSO/ENGANOSO (Confiança: 99.8%)

 Cole ou digite uma notícia aqui: Bolsonaro foi presidente
 Veredito: VERDADEIRO (Confiança: 98.3%)

 Cole ou digite uma notícia aqui: Bolsonaro não é mais presidente
 Veredito: FALSO/ENGANOSO (Confiança: 94.4%)

 Cole ou digite uma notícia aqui: sair
Teste encerrado.
